# ATML PA3 Task 3: Kaggle Runner
Downloads the supplied rollouts from the Google Drive folder linked in the manual, runs the six notebooks in order, and saves them **with their outputs**.

- **Accelerator:** GPU T4 (needed only for notebook 4, the live generation). **Internet:** ON.
- **Run with:** Save Version -> Save & Run All.
- **Afterwards:** download `task3_outputs.zip` from the Output tab and unzip it into `task3_test_time_compute/`. It holds the executed notebooks, results, figures, and split IDs.

The offline notebooks are CPU work. A finished notebook is not re-run, and live generation resumes per question, if you attach a previous version's output as input.

In [ ]:
import glob, os, shutil, subprocess, sys, time
BRANCH = "fix/task1-pipeline"   # Change to "master" once the branch is merged
CODE = "/tmp/ATML-PA3"
OUT = "/kaggle/working/task3_outputs"
DATA = "/tmp/ATML-PA3-Data"
NOTEBOOKS = ["1_data_loading_and_oracle_coverage", "2_quality_scoring_and_rwc", "3_signal_analysis_and_tokenwise_fusion",
             "4_live_sequential_refinement", "5_adaptive_sampling_simulation", "6_compute_allocation_and_failure_analysis"]

shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/moizejaz-31/ATML-PA3.git", CODE], check=True)
subprocess.run('pip install -q -U "math-verify[antlr4_13_2]" gdown pyarrow scipy seaborn "transformers>=4.51" accelerate nbconvert ipykernel', shell=True, check=True)
for version in subprocess.run("pip list 2>/dev/null | grep -iE '^(math-verify|torch|transformers|pyarrow) '", shell=True, capture_output=True, text=True).stdout.splitlines():
    print(version)

In [ ]:
# Restore any previous session, then download the data
previous = sorted(glob.glob("/kaggle/input/**/task3_outputs", recursive=True))
if previous:
    print("Restoring", previous[-1])
    shutil.copytree(previous[-1], OUT, dirs_exist_ok=True)
for folder in ["results", "splits", "notebooks"]:
    os.makedirs(f"{OUT}/{folder}", exist_ok=True)

sys.path.insert(0, CODE)
from task3_test_time_compute.src.data import DRIVE_FOLDER, MATHVISTA_METADATA
import gdown, urllib.request
gdown.download_folder(DRIVE_FOLDER, output=DATA, quiet=False)
urllib.request.urlretrieve(MATHVISTA_METADATA, f"{DATA}/mathvista_testmini.json")
for path in sorted(glob.glob(f"{DATA}/**/*.parquet", recursive=True)):
    print(f"{os.path.getsize(path) / 1e6:8.1f} MB  {path}")

In [ ]:
# Run the notebooks in order. A failure stops here with the error.
env = dict(os.environ, TASK3_DATA_DIR=DATA, TASK3_RESULTS_DIR=f"{OUT}/results", TASK3_SPLITS_DIR=f"{OUT}/splits", TASK3_CACHE_DIR="/tmp/task3_cache")
os.makedirs(env["TASK3_CACHE_DIR"], exist_ok=True)
for name in NOTEBOOKS:
    output = f"{OUT}/notebooks/{name}.ipynb"
    if os.path.exists(output):
        print("Already executed:", name)
        continue
    start = time.time()
    print("Running", name, flush=True)
    result = subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=python3",
                             "--output", f"{name}.ipynb", "--output-dir", "/tmp/task3_executed", f"{CODE}/task3_test_time_compute/notebooks/{name}.ipynb"], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-6000:])
        raise RuntimeError(f"{name} failed")
    shutil.copy(f"/tmp/task3_executed/{name}.ipynb", output)
    print(f"  finished in {(time.time() - start) / 60:.1f} min", flush=True)

In [ ]:
# Package everything to download
subprocess.run(f"rm -f /kaggle/working/task3_outputs.zip && cd {OUT} && zip -q -r /kaggle/working/task3_outputs.zip results splits notebooks", shell=True, check=True)
from IPython.display import Image, display
for path in sorted(glob.glob(f"{OUT}/results/figures/*.png")):
    print(os.path.basename(path))
    display(Image(filename=path, width=900))